# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We first list all RecordSets (tables) provided in the dataset along with their `@id`. For each RecordSet, we also list its fields and columns (all referenced by their `@id`).

In [ ]:
# List record sets and their schema info by @id
record_sets = []

print("RecordSets found in the dataset:")
for record_set in metadata.record_sets:
    print(f"- RecordSet name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    record_sets.append(record_set.id)
    # List fields
    if getattr(record_set, 'fields', None):
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    # List columns (may be synonymous for tabular data)
    if getattr(record_set, 'columns', None):
        print("  Columns:")
        for column in record_set.columns:
            print(f"    - {column.name} (@id: {column.id}, dataType: {getattr(column, 'data_type', None)})")
    print()
print(f"All record_set ids: {record_sets}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Here, we extract all record sets into DataFrames indexed by their `@id`.

In [ ]:
dataframes = {}
for record_set_id in record_sets:
    print(f"Extracting records for {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records loaded for {record_set_id}.")
        dataframes[record_set_id] = pd.DataFrame()


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Below we'll select one RecordSet (by its `@id`) that contains numeric data and explore it. Adjust `selected_record_set_id` and fields as needed based on your dataset structure.

In [ ]:
# Choose a record set with numeric fields for EDA
# You may update this @id after inspecting overview above
if record_sets:
    selected_record_set_id = record_sets[0]
    df = dataframes[selected_record_set_id]
    print(f"Using record set: {selected_record_set_id}")
    
    # Try to pick a numeric column by inspecting the dtypes
    numeric_fields = df.select_dtypes(include=["int", "float"]).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Selected numeric field: {numeric_field}")
        # Filtering based on threshold
        threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field if there is one
        category_fields = df.select_dtypes(include=["object", "category"]).columns.tolist()
        if category_fields:
            group_field = category_fields[0]
            print(f"Grouping by: {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped.head())
        else:
            print("No categorical field found to group by.")
    else:
        print("No numeric fields found in the selected record set.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, you can plot the distribution of the selected numeric field, or compare normalized data across groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only if we have numeric_field and filtered_df
if 'numeric_field' in locals() and not filtered_df.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.show()

    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True)
        plt.title(f"Distribution of normalized {numeric_field}")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.show()

    # If we have a group_field, show group means
    if 'group_field' in locals():
        means = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        plt.figure(figsize=(8,4))
        sns.barplot(data=means, x=group_field, y=numeric_field)
        plt.title(f"Mean of {numeric_field} per {group_field} (filtered records)")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded dataset metadata and overviewed all RecordSets and their fields using the Croissant schema and `mlcroissant`.
- Data from each RecordSet was extracted using the unique `@id` references.
- Basic filtering, normalization, and grouping analyses were performed on numeric data, and the results were visualized.

**Next steps:** Try exploring alternative fields, perform more advanced analyses, or integrate with downstream ML workflows.